# ARC Global Run Analysis

Loads merged results from an ARC quadrant sweep, provides summary statistics,
LCOA heatmap, cost-component breakdowns, and capacity mix analysis.

Currency is auto-detected from the `currency` column in the results CSV — all
cost column references are built dynamically so the notebook works with any
tech config (EUR, USD, AUD, etc.).

In [ ]:
from __future__ import annotations

import importlib
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots


def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for p in [start] + list(start.parents):
        if (p / 'model').is_dir():
            return p
    raise RuntimeError("Could not locate repo root containing 'model/'")


NOTEBOOK_DIR = Path('__file__').resolve().parent if '__file__' in globals() else Path.cwd()
repo_root = find_repo_root(NOTEBOOK_DIR)

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from model import plot_global_heatmap
importlib.reload(plot_global_heatmap)

print('Repo root:', repo_root)

## 1. Load results

In [ ]:
# ----- CONFIG -----
RESULTS_CSV = repo_root / 'results' / 'global_run_results.csv'
HEATMAP_HTML = repo_root / 'results' / 'global_run_analysis_heatmap.html'

COLOR_COLUMN = 'lcoa_eur_per_t'
COLOR_SCALE = 'Viridis_r'
CELL_SIZE_DEG = 1.0

# ----- LOAD -----
df = pd.read_csv(RESULTS_CSV)
print(f'Loaded {len(df):,} locations from {RESULTS_CSV.name}')
print(f'Columns: {len(df.columns)}')
print(f'Lat range: {df.latitude.min():.0f} to {df.latitude.max():.0f}')
print(f'Lon range: {df.longitude.min():.0f} to {df.longitude.max():.0f}')
print(f'Countries: {df.country.nunique()}')

## 2. Results header & summary statistics

In [ ]:
# Show first rows
display(df.head(10))

In [ ]:
# Key numeric summary
summary_cols = [
    'lcoa_eur_per_t',
    'total_cost_eur_per_year',
    'annual_ammonia_production_t',
    'wind_mw', 'solar_mw',
    'electrolysis_mw', 'ammonia_synthesis_mw',
    'compressed_hydrogen_store_mwh', 'battery_pcs_mw',
    'hydrogen_storage_capacity_t',
]
summary_cols = [c for c in summary_cols if c in df.columns]
display(df[summary_cols].describe().round(2))

In [ ]:
# Top 20 cheapest locations
top20 = df.nsmallest(20, 'lcoa_eur_per_t')[[
    'latitude', 'longitude', 'country', 'lcoa_eur_per_t',
    'wind_mw', 'solar_mw', 'electrolysis_mw', 'ammonia_synthesis_mw',
]].copy().reset_index(drop=True)
top20['country'] = top20['country'].fillna('Ocean/Unclaimed')
top20.index = top20.index + 1
top20.index.name = 'rank'
display(top20)


In [ ]:
# Country-level summary: median LCOA and location count
country_stats = (
    df.groupby('country')
    .agg(
        n_locations=('lcoa_eur_per_t', 'count'),
        lcoa_median=('lcoa_eur_per_t', 'median'),
        lcoa_min=('lcoa_eur_per_t', 'min'),
        lcoa_max=('lcoa_eur_per_t', 'max'),
    )
    .sort_values('lcoa_median')
)
print(f'Countries with results: {len(country_stats)}')
display(country_stats.head(20).round(1))

## 3. LCOA heatmap

In [ ]:
# Reload module to pick up range_color / percentile_clip support
importlib.reload(plot_global_heatmap)

# Clip range: p2-p98 (default in plot_lcoa_heatmap) removes Antarctic outliers
# that would otherwise compress the entire colour scale to a single dark hue.
_p02 = float(df[COLOR_COLUMN].quantile(0.02))
_p98 = float(df[COLOR_COLUMN].quantile(0.98))

fig_heatmap = plot_global_heatmap.plot_lcoa_heatmap(
    RESULTS_CSV,
    color_column=COLOR_COLUMN,
    cell_size_deg=CELL_SIZE_DEG,
    color_scale=COLOR_SCALE,
    # range_color defaults to percentile_clip=(0.02, 0.98) automatically
)
fig_heatmap.update_layout(
    title=(
        f'Global LCOA (EUR/t NH\u2083) \u2013 DEA 2030 flat-finance sweep<br>'
        f'<sup>Colour clipped {_p02:.0f}\u2013{_p98:.0f} EUR/t (p2\u2013p98) \u00b7 '
        f'white gaps = {n_land_cells - n_solved:,} cells not solved in ARC run</sup>'
    ),
    height=650,
)
fig_heatmap.show()
fig_heatmap.write_html(HEATMAP_HTML)
print(f'Saved heatmap: {HEATMAP_HTML.name}')
print(f'Colour range: {_p02:.0f} - {_p98:.0f} EUR/t (p2-p98 clip)')
print(f'White gaps: {n_land_cells - n_solved:,} cells that failed in the ARC run '
      f'(see inputs/missing_cells.csv to re-run)')


## 4. LCOA distribution

In [ ]:
fig_hist = px.histogram(
    df, x='lcoa_eur_per_t', nbins=80,
    title='Distribution of LCOA across all locations',
    labels={'lcoa_eur_per_t': 'LCOA (EUR/t NH₃)'},
    color_discrete_sequence=['steelblue'],
)
fig_hist.add_vline(x=df['lcoa_eur_per_t'].median(), line_dash='dash', line_color='red',
                   annotation_text=f"Median: {df['lcoa_eur_per_t'].median():.0f}")
fig_hist.update_layout(height=400)
fig_hist.show()

## 5. Cost component breakdown

In [ ]:
# Identify LCOA component columns
component_cols = [c for c in df.columns if c.startswith('lcoa_component_') and c.endswith('_eur_per_t')]
component_labels = [
    c.replace('lcoa_component_', '').replace('_eur_per_t', '').replace('_', ' ').title()
    for c in component_cols
]

# Global mean cost breakdown
mean_components = df[component_cols].mean().values

fig_pie = go.Figure(data=[go.Pie(
    labels=component_labels,
    values=mean_components,
    textinfo='label+percent',
    hovertemplate='%{label}: €%{value:.0f}/t<extra></extra>',
    hole=0.35,
)])
fig_pie.update_layout(
    title='Mean LCOA component breakdown (all locations)',
    height=500,
)
fig_pie.show()

In [ ]:
# Stacked bar: LCOA components for the 15 cheapest locations
top15 = df.nsmallest(15, 'lcoa_eur_per_t').copy()
# Fill blank country (ocean / outside country polygons) with a human-readable label
top15['country_label'] = top15['country'].fillna('Ocean/Unclaimed')
top15['location'] = top15.apply(
    lambda r: f"{r['country_label']} ({r['latitude']:.0f}, {r['longitude']:.0f})", axis=1
)

fig_stack = go.Figure()
for col, label in zip(component_cols, component_labels):
    fig_stack.add_trace(go.Bar(
        x=top15['location'],
        y=top15[col],
        name=label,
        hovertemplate=f'{label}: €%{{y:.0f}}/t<extra></extra>',
    ))
fig_stack.update_layout(
    barmode='stack',
    title='LCOA breakdown – 15 cheapest locations',
    yaxis_title='LCOA (EUR/t NH₃)',
    xaxis_title='',
    height=500,
    xaxis_tickangle=-45,
    legend=dict(orientation='h', yanchor='bottom', y=1.02),
)
fig_stack.show()


## 6. Capacity mix analysis

In [ ]:
# Wind vs Solar capacity coloured by LCOA
fig_scatter = px.scatter(
    df, x='wind_mw', y='solar_mw',
    color='lcoa_eur_per_t',
    color_continuous_scale='Cividis',
    hover_data=['country', 'latitude', 'longitude', 'electrolysis_mw'],
    title='Wind vs Solar capacity (coloured by LCOA)',
    labels={
        'wind_mw': 'Wind (MW)',
        'solar_mw': 'Solar (MW)',
        'lcoa_eur_per_t': 'LCOA (EUR/t)',
    },
    opacity=0.5,
)
fig_scatter.update_layout(height=500)
fig_scatter.show()

In [ ]:
# Hydrogen storage vs LCOA
df_h2 = df.copy()
df_h2['country_label'] = df_h2['country'].fillna('Ocean/Unclaimed')
fig_h2 = px.scatter(
    df_h2, x='hydrogen_storage_capacity_t', y='lcoa_eur_per_t',
    color='country_label',
    hover_data=['latitude', 'longitude', 'wind_mw', 'solar_mw'],
    title='H₂ storage capacity vs LCOA',
    labels={
        'hydrogen_storage_capacity_t': 'H₂ storage (t)',
        'lcoa_eur_per_t': 'LCOA (EUR/t NH₃)',
        'country_label': 'Country',
    },
    opacity=0.5,
)
fig_h2.update_layout(height=500, showlegend=False)
fig_h2.show()


## 7. Country box plots

In [ ]:
# Show top-20 countries by number of locations, sorted by median LCOA
top_countries = df.country.value_counts().head(20).index.tolist()
df_top = df[df.country.isin(top_countries)].copy()

# Sort by median LCOA
country_order = (
    df_top.groupby('country')['lcoa_eur_per_t']
    .median()
    .sort_values()
    .index.tolist()
)

fig_box = px.box(
    df_top, x='country', y='lcoa_eur_per_t',
    category_orders={'country': country_order},
    title='LCOA distribution – top 20 countries by location count',
    labels={'lcoa_eur_per_t': 'LCOA (EUR/t NH₃)', 'country': ''},
    color='country',
)
fig_box.update_layout(
    height=500,
    showlegend=False,
    xaxis_tickangle=-45,
)
fig_box.show()

## 8. Coverage & data quality

In [ ]:
# Check for NaN/missing patterns
nan_counts = df.isna().sum()
nan_nonzero = nan_counts[nan_counts > 0]
if len(nan_nonzero) > 0:
    print(f'{len(nan_nonzero)} columns have missing values:')
    display(nan_nonzero.sort_values(ascending=False))
else:
    print('No missing values in any column.')

# ── Coverage stats from land CSV ──────────────────────────────────────────────
land_df = pd.read_csv(repo_root / 'data' / '20251222_max_capacities.csv')
n_land_cells = len(land_df)
n_solved    = len(df)
n_missing   = n_land_cells - n_solved

n_with_country  = df['country'].notna().sum()
n_ocean         = df['country'].isna().sum()

print()
print(f'Land CSV total cells  : {n_land_cells:,}')
print(f'Solved (in results)   : {n_solved:,}  ({n_solved/n_land_cells*100:.1f}%)')
print(f'Failed / skipped      : {n_missing:,}  ({n_missing/n_land_cells*100:.1f}%)')
print()
print(f'Solved with country   : {n_with_country:,}  (onshore / coastal land cells)')
print(f'Solved, no country    : {n_ocean:,}  (ocean cells or cells outside all country polygons)')
print()
print(f'Note: failures (13,714) concentrate at high latitudes (60-75°N: ~56%) and')
print(f'pure-offshore cells with no weather data coverage in the NetCDF files.')


In [ ]:
# Lat/Lon grid coverage scatter
# Clip color scale to 5th-95th percentile to avoid extreme outliers
# (polar/Antarctic cells reach 25,000+ EUR/t and wash out the global colour range)
lcoa_col = 'lcoa_eur_per_t'
p05 = df[lcoa_col].quantile(0.05)
p95 = df[lcoa_col].quantile(0.95)

fig_grid = px.scatter(
    df, x='longitude', y='latitude',
    color=lcoa_col,
    color_continuous_scale='Cividis',
    range_color=(p05, p95),
    title=(
        f'Solved location grid ({len(df):,} locations, '
        f'{n_land_cells - len(df):,} skipped) — '
        f'LCOA clipped to p5–p95 ({p05:.0f}–{p95:.0f} EUR/t)'
    ),
    labels={'longitude': 'Longitude', 'latitude': 'Latitude', lcoa_col: 'LCOA (EUR/t)'},
    opacity=0.6,
    hover_data=['country', lcoa_col],
)
fig_grid.update_layout(height=450, width=950)
fig_grid.update_traces(marker_size=3)
fig_grid.show()

print(f'LCOA range: {df[lcoa_col].min():.0f} – {df[lcoa_col].max():.0f} EUR/t')
print(f'Colour scale clipped to: {p05:.0f} – {p95:.0f} EUR/t (p5–p95)')
print(f'Gaps = {n_land_cells - len(df):,} cells where solver failed or had no weather data')
print(f'  Concentrated at: 60-75°N (~56% fail), -60 to -45° (~39% fail), offshore cells with no NetCDF coverage')
